# Instructional Notebook for SHRED + Biomechanics

The following lines can be uncommented if running this notebook in Google Colab. Uncomment by highlighting lines and pressing Ctrl+/

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')
#!pip install mat73
#!git clone https://github.com/Jan-Williams/pyshred
#%cd /content/pyshred

These lines import standard packages for managing data.

In [3]:
import os
import numpy as np
import altair as alt
import pandas as pd
from processdata import TimeSeriesDataset
import models_TCN
import torch
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
import mat73
import functions as ft

# *** Update ***
Load a subject's data and manipulate the dataframe to be "tidy" = one row per time step and one column per signal. Depending on the dataset, loading and managing data will look different.

## Obtain subject's experimental data

Items to adjust before running a trial:

*   save_df - True (save) or False (don't save)
*   subject - '##'
*   activity code - AC## (this may need a different identifier depending on the dataset, just need a way to distinguish running speeds)

In [4]:
# Change subject number
subj = '01'    # 01-09
save_df = True   # True: save SHRED output dataframes only, False: don't save
trial_length = 5  # in minutes, accepts integers 1-6
frequency = 128 # in Hz, accepts integers up to 128

# adjust file path for saving if parameters are modified from 6min or 128Hz
if trial_length == 6:
    save_tag = str(frequency)+'Hz'
elif frequency == 128:
    save_tag = str(trial_length)+'min'


## Import Matlab file structure with subject's experimental data

Access directories where data is stored and will be saved. Manually set up folders before running the code block to ensure known file paths.

In [5]:
cwd = os.getcwd()
main_path = os.path.dirname(cwd) + '/Datasets' 

# Alternatively, use the below lines if using Colab
# main_path = '/content/drive/MyDrive/Colab_Notebooks/Datasets'
# main_path = cwd+'/Datasets'

dataset_path = main_path+'/Data' # sets path to dataset / raw data
dataframe_path = main_path+'/Dataframes'  # file path for saved dataframe results of test data
figure_path = main_path+'/Figures'
model_path = main_path+'/Models' # optionally, save the models that are trained
print(dataset_path)
print(dataframe_path)


/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Data
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes


Note on changing file directory and needing to save the parent directory:

https://stackoverflow.com/questions/14462833/how-can-i-go-back-to-the-previous-working-directory-after-changing-it

In [6]:
# load .mat file into pandas dataframe
load_mat = mat73.loadmat(dataset_path+'/Subject'+subj+'.mat')['Subject'+subj]
df = pd.DataFrame.from_dict(load_mat)

The example dataset contains several activities, two of which are 'Walking' and 'Running'. Access each independently.

In [7]:
#df_tmp = pd.DataFrame(data=df['Walking']['APDM_Accel']['Data'],
#                      columns = df['Walking']['APDM_Accel']['Labels'])

df_tmp = pd.DataFrame(data=df['Running']['APDM_Accel']['Data'],
                      columns = df['Running']['APDM_Accel']['Labels'])

pd.set_option('display.max_columns', None)

df_tmp.tail(5) # check that correct data was selected

Time (s) Activity Code                Waist                      \
                                  Acceleration (m/s^2)                       
                                                     x         y         z   
231348  1807.343463          12.0            -9.353293  1.222394  1.625345   
231349  1807.351275          12.0            -9.335909  1.123459  1.540691   
231350  1807.359088          12.0            -9.171153  1.077339  1.206564   
231351  1807.366900          12.0            -9.017789  1.329124  1.277886   
231352  1807.374712          12.0            -8.865735  1.374722  1.454045   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
231348                -0.142169  0.034614 -0.177125           50.843437   
231349                -0.148348  0.037640 -0.184895           50.714705   
231350                -0.128062  0.063893 -0.182946           50.640699   
231351                -0.073453  0.047206 -0.184080           50.525998   
231352                 0.042053  0.055382 -0.189070           50.544571   

                                            Chest                      \
                             Acceleration (m/s^2)                       
                y          z                    x         y         z   
231348  16.872015  10.648010            -9.088658  0.150357  1.608253   
231349  17.094845  10.726108            -8.898073  0.133779  1.682710   
231350  17.056767  10.871868            -8.852520  0.135237  1.756955   
231351  17.037792  10.712959            -8.971994 -0.000197  1.824701   
231352  17.160503  10.715923            -8.925372 -0.223933  1.943991   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
231348                 0.440451  0.272910  0.389406           57.418567   
231349                 0.425002  0.231876  0.373033           57.305999   
231350                 0.397494  0.157865  0.343987           57.243980   
231351                 0.355106  0.073336  0.299004           57.061209   
231352                 0.329611 -0.029720  0.263512           56.825190   

                                      Left Ankle                      \
                            Acceleration (m/s^2)                       
               y          z                    x         y         z   
231348  1.518668  14.197069           -10.376046 -0.368902 -2.140184   
231349  1.607555  14.145851           -10.403057 -0.227233 -2.169853   
231350  1.473774  14.260109           -10.562581 -0.652918 -2.124238   
231351  1.496668  14.406160           -10.797644 -0.895918 -1.633860   
231352  1.494085  14.216662           -10.658891 -0.008163 -1.353547   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
231348                 0.639215 -0.079757 -0.241094           32.060207   
231349                 0.665515 -0.081127 -0.240671           31.953617   
231350                 0.692417 -0.089017 -0.254645           31.504868   
231351                 0.726948 -0.105717 -0.295743           31.344335   
231352                 0.788216 -0.114308 -0.310793           31.309810   

                                     Right Ankle                      \
                            Acceleration (m/s^2)                       
                y         z                    x         y         z   
231348  21.965298 -1.408134           -13.431479  1.032572 -0.510666   
231349  22.193238 -1.619954           -14.667708  1.500816 -1.404407   
231350  22.171354 -1.640882           -15.673375  2.739823 -2

In [8]:
# remove units and simplify column titles
columns_str = ["_".join(df_tmp.columns[i]).replace(" ", "") for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = columns_str

l_replace = [df_tmp.columns[i].replace('(m/s^2)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(rad/s)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(uT)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace
df_tmp.head()

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,LeftFoot_Acceleration_x,LeftFoot_Acceleration_y,LeftFoot_Acceleration_z,LeftFoot_AngularVelocity_x,LeftFoot_AngularVelocity_y,LeftFoot_AngularVelocity_z,LeftFoot_MagneticField_x,LeftFoot_MagneticField_y,LeftFoot_MagneticField_z,RightFoot_Acceleration_x,RightFoot_Acceleration_y,RightFoot_Acceleration_z,RightFoot_AngularVelocity_x,RightFoot_AngularVelocity_y,RightFoot_AngularVelocity_z,RightFoot_MagneticField_x,RightFoot_MagneticField_y,RightFoot_MagneticField_z
0,0.007812,22.0,-9.846344,0.171822,1.227401,0.091794,-0.027342,-0.014275,46.815190,16.071594,13.902695,-9.764181,1.311812,0.985503,0.070709,0.084543,0.012162,55.879245,-1.209833,11.792871,-9.741885,-0.468814,-0.882530,0.118064,0.017438,0.021202,31.514212,18.934442,-8.637949,-9.838516,-0.574557,-1.155564,-0.006665,0.099050,0.009824,24.682745,-21.892916,18.683383,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.015624,22.0,-9.775612,0.050246,1.209962,0.094758,-0.032213,-0.006748,46.996207,16.216266,13.923646,-9.757018,1.316749,0.992521,0.067192,0.088808,0.013694,55.828066,-1.158919,11.847663,-9.729948,-0.520553,-0.887037,0.113823,0.015991,0.017926,31.112110,18.884349,-8.560825,-9.851844,-0.576608,-1.153336,-0.003499,0.100780,0.011414,24.586635,-21.917585,18.649075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.023437,22.0,-9.835819,0.125845,1.167702,0.097836,-0.038619,-0.010032,46.728334,16.212284,13.792188,-9.754793,1.332469,0.997156,0.065436,0.091731,0.010487,55.859044,-1.419488,11.842671,-9.728399,-0.466783,-0.891656,0.116528,0.015885,0.019570,31.061167,19.012600,-8.561372,-9.856227,-0.571967,-1.148940,-0.003592,0.095519,0.009830,24.610019,-21.911564,18.789673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.031249,22.0,-9.810564,0.107856,1.203309,0.086923,-0.035422,-0.006915,46.646590,16.073072,13.775756,-9.757444,1.331871,0.987872,0.081388,0.086023,0.012167,55.757635,-1.280665,11.819464,-9.723669,-0.453419,-0.884954,0.113933,0.020142,0.017943,31.082365,19.015587,-8.637877,-9.849360,-0.574299,-1.151208,-0.002224,0.080491,0.011503,24.576289,-21.911981,18.789370,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.039061,22.0,-9.800482,0.050737,1.143245,0.091569,-0.041706,-0.007019,46.688921,16.195991,13.996201,-9.746306,1.337988,0.989957,0.081584,0.084634,0.013785,55.632360,-1.164697,11.939039,-9.716266,-0.484920,-0.887274,0.108888,0.018438,0.019454,31.019347,18.862127,-8.912165,-9.845130,-0.569882,-1.144508,-0.008403,0.088848,0.009861,24.728641,-21.912624,18.651281,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Obtain data with desired activity code

In [9]:
# subject running codes [12, 13, 14] = 1.8, 2,2, 2.7 m/s
AC_path = 'AC1214'
df_1=df_tmp.loc[df_tmp['ActivityCode__']==13].dropna(axis=1,how='all')
df_2=df_tmp.loc[df_tmp['ActivityCode__']==14].dropna(axis=1,how='all')
df_3=df_tmp.loc[df_tmp['ActivityCode__']==12].dropna(axis=1,how='all')
df_2

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
138245,1080.004687,14.0,-3.372638,0.233544,0.018923,0.670585,-0.370853,0.405388,49.542897,21.226252,16.426046,-6.106585,2.337556,-0.803689,0.323016,0.398863,0.177662,53.510772,-7.197517,14.886496,-16.246151,-11.460346,-2.774555,-1.050385,0.966521,8.558224,32.091489,26.618803,-3.663403,-11.034618,-12.400804,-8.939685,-1.308537,-1.056722,0.768119,14.157091,-33.913572,16.090631
138246,1080.012500,14.0,-2.162693,-0.328624,-0.704124,0.444071,-0.384835,0.442240,49.616865,21.126525,16.538396,-4.450399,0.973722,-1.219477,0.284850,0.199353,0.251566,53.575014,-7.068218,15.017490,-19.663835,-16.320841,-3.723295,-0.907463,0.892241,8.557838,34.136118,24.621912,-3.044350,-7.554622,-8.882235,-9.565899,-0.499558,-0.719613,1.083773,14.339057,-34.060772,16.016379
138247,1080.020312,14.0,-1.223847,-0.551038,-1.034889,0.276061,-0.440277,0.562511,49.554258,20.914726,16.504507,-2.547924,0.399285,-1.793944,0.378271,0.088125,0.490292,53.567330,-7.037938,14.869948,-24.017165,-21.433318,-3.289888,-0.711916,0.809143,8.431439,35.945134,22.400249,-2.683516,-6.808484,-4.507297,-8.244035,-0.073092,-0.272298,1.575430,14.577229,-34.022331,15.674487
138248,1080.028124,14.0,-0.227446,-0.226153,-1.227845,0.214435,-0.445822,0.662766,49.798805,20.922483,16.577022,-2.031135,0.128477,-1.849271,0.452199,-0.079791,0.669848,53.540525,-7.324864,14.819242,-25.279244,-25.597349,-2.262449,-0.852510,0.883626,8.112348,37.452894,19.824997,-2.418215,-7.623908,-1.499453,-6.226309,-0.251172,0.016850,2.183657,14.430384,-34.101938,15.433980
138249,1080.035936,14.0,0.956586,0.376656,-1.349267,0.204588,-0.533824,0.714251,49.881881,20.510637,16.920124,-1.237608,1.530044,-1.561266,0.610862,-0.272398,0.768427,53.700915,-7.617658,15.233412,-23.868389,-28.048991,-2.011452,-1.356068,1.012450,7.530704,38.792423,17.433467,-2.060894,-9.271189,0.305924,-3.795190,-0.770712,0.186915,2.807590,14.334634,-34.364764,15.613160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184322,1439.967189,14.0,-18.666289,-4.500684,-1.101372,0.911143,-1.292925,0.494124,45.091387,16.621520,26.290780,1.895743,-10.625612,2.051075,-5.736320,-0.234091,1.056988,57.026064,-0.072214,11.223191,-13.669348,7.786734,-2.855058,-2.337095,-1.030070,-2.680430,8.294447,32.887510,-13.933101,11.958795,1.297591,9.045534,-6.746858,-0.010117,4.900339,26.071383,-17.137099,21.985465
184323,1439.975001,14.0,-18.561967,-0.191393,1.650342,2.579193,0.423558,0.615597,45.906280,16.677853,26.699735,0.594152,-3.443288,2.268143,-5.400404,0.069034,0.778716,57.085527,-0.652228,10.476547,-9.419378,3.286309,-2.289718,-2.450800,-0.945640,-1.964250,7.646038,33.552008,-13.324461,15.637125,1.796823,-3.901085,-1.742367,1.313201,5.270676,25.417715,-18.744092,21.764860
184324,1439.982813,14.0,-26.645392,2.946783,0.262649,2.010343,1.163141,-0.625906,47.195223,16.812079,26.615249,-2.821267,-7.303968,1.856312,-4.161994,0.107735,-0.045883,57.018347,-1.592320,9.801052,-9.730954,1.049938,-2.145450,-2.338991,-0.867197,-1.303754,6.650909,34.110605,-12.826474,-1.137

### Simplify dataframe

In [10]:
# trim length of trial (number of rows in df)
obs_samples_trial = trial_length*60*frequency
df_1 = df_1.tail(obs_samples_trial)
df_2 = df_2.tail(obs_samples_trial) # keep last n samples to exclude speed transitions
df_3 = df_3.tail(obs_samples_trial)

# downsample trial
obs_samples_freq = int(128/frequency)

df_1 = df_1.iloc[::obs_samples_freq,:]
df_2 = df_2.iloc[::obs_samples_freq,:]
df_3 = df_3.iloc[::obs_samples_freq,:]
df_2

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
145927,1140.017968,14.0,-21.507674,2.225995,1.571816,-1.764344,1.457840,0.078401,46.999161,16.057122,17.014367,-15.470546,7.129028,-0.194896,-1.246388,0.638730,-0.959855,55.030424,-0.948900,17.512487,-18.895170,-13.371935,-3.854182,1.177129,0.601539,-2.837383,32.766473,24.993452,3.195923,-2.454929,-2.913935,-2.549265,2.143909,-0.828032,-0.435036,-3.424384,-42.288182,15.099166
145928,1140.025780,14.0,-24.646400,1.908350,-0.562389,-1.488918,1.200035,0.147073,47.506316,16.095249,17.045009,-18.916291,1.432432,-0.621804,-0.980632,0.440489,-1.149383,55.103795,-0.752654,17.625991,-8.664984,-1.354278,0.561449,1.296216,0.781695,-2.129523,31.475687,26.175530,3.250227,-1.751772,-3.156174,-2.862659,1.824526,-0.845998,-1.249249,-3.192449,-42.199174,15.410833
145929,1140.033592,14.0,-19.178551,-0.773347,-2.844979,-0.743644,0.966679,0.002972,47.799287,16.106041,17.012386,-21.608185,-2.957324,-0.437966,-0.519623,0.219935,-1.179284,54.975985,-0.742430,17.597781,-10.183516,-3.925713,1.675239,0.743880,0.772043,-2.276361,30.722228,26.950706,3.139523,-2.459084,-2.080640,-2.635835,1.551726,-0.911412,-2.064687,-3.068192,-41.983848,15.925432
145930,1140.041405,14.0,-24.347215,5.328815,3.041549,-0.904848,0.735307,0.026376,47.697627,16.033202,17.158450,-23.156364,-6.312682,-0.676571,-0.178481,0.027462,-0.831683,54.940309,-0.415764,17.625151,-11.901889,-11.632171,-1.047341,0.647351,0.675690,-2.127828,30.090535,27.325300,2.973068,-2.823463,-0.078233,-1.952238,1.255387,-0.999577,-2.883094,-2.712641,-41.916498,16.452965
145931,1140.049217,14.0,-24.696106,5.535976,-4.063290,-0.961385,0.106615,0.027554,47.960065,15.817000,17.794345,-23.705403,-8.416875,-0.829783,0.157595,-0.094760,-0.179931,55.242472,-0.374707,17.506222,-11.420449,-9.507497,-4.694069,0.982234,0.553377,-1.710922,28.936799,27.700682,3.267332,-2.598315,1.403085,-1.209009,0.925166,-1.076483,-3.631692,-2.215133,-41.827573,16.721821
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184322,1439.967189,14.0,-18.666289,-4.500684,-1.101372,0.911143,-1.292925,0.494124,45.091387,16.621520,26.290780,1.895743,-10.625612,2.051075,-5.736320,-0.234091,1.056988,57.026064,-0.072214,11.223191,-13.669348,7.786734,-2.855058,-2.337095,-1.030070,-2.680430,8.294447,32.887510,-13.933101,11.958795,1.297591,9.045534,-6.746858,-0.010117,4.900339,26.071383,-17.137099,21.985465
184323,1439.975001,14.0,-18.561967,-0.191393,1.650342,2.579193,0.423558,0.615597,45.906280,16.677853,26.699735,0.594152,-3.443288,2.268143,-5.400404,0.069034,0.778716,57.085527,-0.652228,10.476547,-9.419378,3.286309,-2.289718,-2.450800,-0.945640,-1.964250,7.646038,33.552008,-13.324461,15.637125,1.796823,-3.901085,-1.742367,1.313201,5.270676,25.417715,-18.744092,21.764860
184324,1439.982813,14.0,-26.645392,2.946783,0.262649,2.010343,1.163141,-0.625906,47.195223,16.812079,26.615249,-2.821267,-7.303968,1.856312,-4.161994,0.107735,-0.045883,57.018347,-1.592320,9.801052,-9.730954,1.049938,-2.145450,-2.338991,-0.867197,-1.303754,6.650909,34.110605,-12.82647

Only include sensor data for model training and testing; remove time and activity code columns

In [11]:
df_1_data = df_1.iloc[:,2:] 
df_2_data = df_2.iloc[:,2:] 
df_3_data = df_3.iloc[:,2:] 
df_2_data

,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
145927,-21.507674,2.225995,1.571816,-1.764344,1.457840,0.078401,46.999161,16.057122,17.014367,-15.470546,7.129028,-0.194896,-1.246388,0.638730,-0.959855,55.030424,-0.948900,17.512487,-18.895170,-13.371935,-3.854182,1.177129,0.601539,-2.837383,32.766473,24.993452,3.195923,-2.454929,-2.913935,-2.549265,2.143909,-0.828032,-0.435036,-3.424384,-42.288182,15.099166
145928,-24.646400,1.908350,-0.562389,-1.488918,1.200035,0.147073,47.506316,16.095249,17.045009,-18.916291,1.432432,-0.621804,-0.980632,0.440489,-1.149383,55.103795,-0.752654,17.625991,-8.664984,-1.354278,0.561449,1.296216,0.781695,-2.129523,31.475687,26.175530,3.250227,-1.751772,-3.156174,-2.862659,1.824526,-0.845998,-1.249249,-3.192449,-42.199174,15.410833
145929,-19.178551,-0.773347,-2.844979,-0.743644,0.966679,0.002972,47.799287,16.106041,17.012386,-21.608185,-2.957324,-0.437966,-0.519623,0.219935,-1.179284,54.975985,-0.742430,17.597781,-10.183516,-3.925713,1.675239,0.743880,0.772043,-2.276361,30.722228,26.950706,3.139523,-2.459084,-2.080640,-2.635835,1.551726,-0.911412,-2.064687,-3.068192,-41.983848,15.925432
145930,-24.347215,5.328815,3.041549,-0.904848,0.735307,0.026376,47.697627,16.033202,17.158450,-23.156364,-6.312682,-0.676571,-0.178481,0.027462,-0.831683,54.940309,-0.415764,17.625151,-11.901889,-11.632171,-1.047341,0.647351,0.675690,-2.127828,30.090535,27.325300,2.973068,-2.823463,-0.078233,-1.952238,1.255387,-0.999577,-2.883094,-2.712641,-41.916498,16.452965
145931,-24.696106,5.535976,-4.063290,-0.961385,0.106615,0.027554,47.960065,15.817000,17.794345,-23.705403,-8.416875,-0.829783,0.157595,-0.094760,-0.179931,55.242472,-0.374707,17.506222,-11.420449,-9.507497,-4.694069,0.982234,0.553377,-1.710922,28.936799,27.700682,3.267332,-2.598315,1.403085,-1.209009,0.925166,-1.076483,-3.631692,-2.215133,-41.827573,16.721821
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184322,-18.666289,-4.500684,-1.101372,0.911143,-1.292925,0.494124,45.091387,16.621520,26.290780,1.895743,-10.625612,2.051075,-5.736320,-0.234091,1.056988,57.026064,-0.072214,11.223191,-13.669348,7.786734,-2.855058,-2.337095,-1.030070,-2.680430,8.294447,32.887510,-13.933101,11.958795,1.297591,9.045534,-6.746858,-0.010117,4.900339,26.071383,-17.137099,21.985465
184323,-18.561967,-0.191393,1.650342,2.579193,0.423558,0.615597,45.906280,16.677853,26.699735,0.594152,-3.443288,2.268143,-5.400404,0.069034,0.778716,57.085527,-0.652228,10.476547,-9.419378,3.286309,-2.289718,-2.450800,-0.945640,-1.964250,7.646038,33.552008,-13.324461,15.637125,1.796823,-3.901085,-1.742367,1.313201,5.270676,25.417715,-18.744092,21.764860
184324,-26.645392,2.946783,0.262649,2.010343,1.163141,-0.625906,47.195223,16.812079,26.615249,-2.821267,-7.303968,1.856312,-4.161994,0.107735,-0.045883,57.018347,-1.592320,9.801052,-9.730954,1.049938,-2.145450,-2.338991,-0.867197,-1.303754,6.650909,34.110605,-12.826474,-1.137238,6.171991,-12.454241,0.468594,1.998434,3.367290,24.988958,-21.129452,21.620725
184325,-17.582697,3.638437,-0.516719,2.795161,1.156525,-1.652173,48.668526,17.3

In [ ]:
# convert pandas dataframe to numpy array
load_X = df_2_data.to_numpy()
#load_X = np.concatenate((df_2_data, df_1_data), axis=0)
load_XT = df_3_data.to_numpy()
load_X.shape, load_XT.shape

((76800, 36), (38400, 36))

In [36]:
df_3_data.head()

,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
192953,0.669899,-2.223057,-1.920976,0.382160,0.018951,-0.190242,46.818495,16.133455,20.317440,-0.608555,2.186329,-2.453348,-1.200798,-0.190891,-0.859238,56.914883,-0.147788,13.506177,-16.837039,-1.113608,-3.173914,1.443275,-0.813301,-4.573894,16.201047,31.903772,-10.239838,-5.769559,16.215272,-29.334027,1.384635,1.170269,2.327332,29.566980,-10.954166,19.502649
192954,-0.387656,-2.260268,-1.068502,0.639635,-0.029822,-0.200912,46.640298,16.126301,20.766806,-2.552028,4.733920,-2.715482,-1.076356,-0.460825,-0.816853,56.892127,0.018744,13.332674,-18.180350,-1.333923,-5.275768,1.345168,-0.685515,-4.528758,15.318438,32.378329,-10.603190,-18.557201,4.409515,-34.629938,2.015977,1.830057,3.041721,29.150574,-11.261101,19.603632
192955,-2.585622,-2.271082,0.088246,0.856521,0.108481,-0.191238,46.356683,15.874559,20.944286,-6.018570,10.039680,-1.623083,-0.829050,-0.968216,-0.747826,57.124147,0.178464,12.907803,-19.704526,-0.715439,-7.125071,1.538587,-0.611216,-4.343658,14.333629,32.936853,-10.988315,-30.034481,-10.414625,-30.541549,2.010379,2.409682,3.639009,28.899816,-12.045292,19.701434
192956,-5.932279,-2.450753,0.260666,0.668717,0.346053,-0.094937,46.105430,15.410128,21.634286,-16.160978,19.740903,-0.448068,-0.537471,-1.498537,-0.734937,57.185280,0.359790,12.533050,-23.443022,1.174618,-5.585193,1.596861,-0.629626,-4.046760,13.182506,33.231829,-11.336208,-37.768251,1.329246,1.684808,1.457121,2.537937,4.997303,28.645632,-12.605279,20.523114
192957,-9.223000,-2.557179,-2.473370,0.804632,0.472711,0.098260,45.845252,15.745351,22.661836,-33.055878,24.773968,1.431991,-0.739346,-1.419947,-1.122380,57.261410,0.430477,12.143817,-24.260333,5.909558,-2.813163,0.799411,-0.765023,-3.706661,12.350512,33.585000,-11.843860,-41.018567,28.727346,21.823860,0.145407,2.984106,6.224279,28.220966,-13.209799,21.069269


In [34]:
df_2_data.head()

,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
145927,-21.507674,2.225995,1.571816,-1.764344,1.457840,0.078401,46.999161,16.057122,17.014367,-15.470546,7.129028,-0.194896,-1.246388,0.638730,-0.959855,55.030424,-0.948900,17.512487,-18.895170,-13.371935,-3.854182,1.177129,0.601539,-2.837383,32.766473,24.993452,3.195923,-2.454929,-2.913935,-2.549265,2.143909,-0.828032,-0.435036,-3.424384,-42.288182,15.099166
145928,-24.646400,1.908350,-0.562389,-1.488918,1.200035,0.147073,47.506316,16.095249,17.045009,-18.916291,1.432432,-0.621804,-0.980632,0.440489,-1.149383,55.103795,-0.752654,17.625991,-8.664984,-1.354278,0.561449,1.296216,0.781695,-2.129523,31.475687,26.175530,3.250227,-1.751772,-3.156174,-2.862659,1.824526,-0.845998,-1.249249,-3.192449,-42.199174,15.410833
145929,-19.178551,-0.773347,-2.844979,-0.743644,0.966679,0.002972,47.799287,16.106041,17.012386,-21.608185,-2.957324,-0.437966,-0.519623,0.219935,-1.179284,54.975985,-0.742430,17.597781,-10.183516,-3.925713,1.675239,0.743880,0.772043,-2.276361,30.722228,26.950706,3.139523,-2.459084,-2.080640,-2.635835,1.551726,-0.911412,-2.064687,-3.068192,-41.983848,15.925432
145930,-24.347215,5.328815,3.041549,-0.904848,0.735307,0.026376,47.697627,16.033202,17.158450,-23.156364,-6.312682,-0.676571,-0.178481,0.027462,-0.831683,54.940309,-0.415764,17.625151,-11.901889,-11.632171,-1.047341,0.647351,0.675690,-2.127828,30.090535,27.325300,2.973068,-2.823463,-0.078233,-1.952238,1.255387,-0.999577,-2.883094,-2.712641,-41.916498,16.452965
145931,-24.696106,5.535976,-4.063290,-0.961385,0.106615,0.027554,47.960065,15.817000,17.794345,-23.705403,-8.416875,-0.829783,0.157595,-0.094760,-0.179931,55.242472,-0.374707,17.506222,-11.420449,-9.507497,-4.694069,0.982234,0.553377,-1.710922,28.936799,27.700682,3.267332,-2.598315,1.403085,-1.209009,0.925166,-1.076483,-3.631692,-2.215133,-41.827573,16.721821


In [37]:
load_XT

array([[  0.669899,  -2.223057,  -1.920976, ...,  29.56698 , -10.954166,
         19.502649],
       [ -0.387656,  -2.260268,  -1.068502, ...,  29.150574, -11.261101,
         19.603632],
       [ -2.585622,  -2.271082,   0.088246, ...,  28.899816, -12.045292,
         19.701434],
       ...,
       [ -9.171153,   1.077339,   1.206564, ...,  25.459896, -28.923799,
         14.199359],
       [ -9.017789,   1.329124,   1.277886, ...,  26.471377, -27.337109,
         14.305298],
       [ -8.865735,   1.374722,   1.454045, ...,  27.10479 , -25.66672 ,
         14.51871 ]], shape=(38400, 36))

## Set up sensors

In [13]:
from random import choice

lags = frequency # length of trajectory used to train LSTM; chose 128 for Ingraham data sampled at 128 Hz
n = load_X.shape[0] # total number of time steps (observations)
m = load_X.shape[1] # number of features per time step

time = np.arange(1, n+1, 1)

## Visualize IMU data

Observing raw data is important for understanding what is being used to train and test models. We visualize data using altair (alt). Two tutorials on some basic functionality are linked below:

* Long tutorial (1hr): https://youtu.be/umTwkgQoo_E

* Short tutorial (20min): https://youtu.be/o-nVM_FdIVc

Uncomment the line below when code is fully functioning to disable the 5000-row dataframe limit

In [14]:
# alt.data_transformers.disable_max_rows()

In [15]:
# set time to start at 0 (optional for clean viz)
time_zeroed = df_2.loc[:,"Time(s)__"] - df_2["Time(s)__"].iloc[0]

# view the first portion of the trial
df_2_data_reduced = df_2.head(1000)
df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)

# view the last portion of the trial
#df_2_data_reduced = df_2.tail(4000) 
#df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.tail(4000)

df_2_data_reduced.head()

/tmp/ipykernel_2918717/3481633552.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)


,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,Time_Zeroed(s)
145927,1140.017968,14.0,-21.507674,2.225995,1.571816,-1.764344,1.457840,0.078401,46.999161,16.057122,17.014367,-15.470546,7.129028,-0.194896,-1.246388,0.638730,-0.959855,55.030424,-0.948900,17.512487,-18.895170,-13.371935,-3.854182,1.177129,0.601539,-2.837383,32.766473,24.993452,3.195923,-2.454929,-2.913935,-2.549265,2.143909,-0.828032,-0.435036,-3.424384,-42.288182,15.099166,0.000000
145928,1140.025780,14.0,-24.646400,1.908350,-0.562389,-1.488918,1.200035,0.147073,47.506316,16.095249,17.045009,-18.916291,1.432432,-0.621804,-0.980632,0.440489,-1.149383,55.103795,-0.752654,17.625991,-8.664984,-1.354278,0.561449,1.296216,0.781695,-2.129523,31.475687,26.175530,3.250227,-1.751772,-3.156174,-2.862659,1.824526,-0.845998,-1.249249,-3.192449,-42.199174,15.410833,0.007812
145929,1140.033592,14.0,-19.178551,-0.773347,-2.844979,-0.743644,0.966679,0.002972,47.799287,16.106041,17.012386,-21.608185,-2.957324,-0.437966,-0.519623,0.219935,-1.179284,54.975985,-0.742430,17.597781,-10.183516,-3.925713,1.675239,0.743880,0.772043,-2.276361,30.722228,26.950706,3.139523,-2.459084,-2.080640,-2.635835,1.551726,-0.911412,-2.064687,-3.068192,-41.983848,15.925432,0.015624
145930,1140.041405,14.0,-24.347215,5.328815,3.041549,-0.904848,0.735307,0.026376,47.697627,16.033202,17.158450,-23.156364,-6.312682,-0.676571,-0.178481,0.027462,-0.831683,54.940309,-0.415764,17.625151,-11.901889,-11.632171,-1.047341,0.647351,0.675690,-2.127828,30.090535,27.325300,2.973068,-2.823463,-0.078233,-1.952238,1.255387,-0.999577,-2.883094,-2.712641,-41.916498,16.452965,0.023437
145931,1140.049217,14.0,-24.696106,5.535976,-4.063290,-0.961385,0.106615,0.027554,47.960065,15.817000,17.794345,-23.705403,-8.416875,-0.829783,0.157595,-0.094760,-0.179931,55.242472,-0.374707,17.506222,-11.420449,-9.507497,-4.694069,0.982234,0.553377,-1.710922,28.936799,27.700682,3.267332,-2.598315,1.403085,-1.209009,0.925166,-1.076483,-3.631692,-2.215133,-41.827573,16.721821,0.031249


Select which sensor location to visualize.

In [16]:
location = 'RightAnkle' # RightAnkle, LeftAnkle, Chest, Waist

In [17]:
# Define signal types and axes.
sensor = ['Acceleration', 'AngularVelocity', 'MagneticField']
dir = ['x','y','z']
plotStack = [0,0,0] # Preallocate plot for each signal

# Generate plots for each sensor type
for iSensor in range(len(sensor)): # loop through the signal types
    # create plots for x,y,z directions
    x_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_x', title = sensor[iSensor]),
        color = alt.value('#c6dbef')
    ).properties(
        width = 1000,
        height = 200
    )
    y_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_y', title = sensor[iSensor]),
        color = alt.value("#6baed6")
    )
    z_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_z', title = sensor[iSensor]),
        color = alt.value("#08519c")
    ).interactive()
    # Combine x,y,z plots
    plotStack[iSensor] = x_signal + y_signal + z_signal

alt.vconcat(plotStack[0], plotStack[1], plotStack[2]).properties(title = [location,""])

alt.VConcatChart(...)

# SHRED model function

In [18]:
### Generate input sequences to a SHRED model
def train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags):
  """
    Trains SHRED model for time series reconstruction

  Args: 
    transformed_X (numpy array): MinMax scaled dataset.
    sc (MinMaxScaler): Fitted MinMax scaler for inverse transformation.
    train_indices (array): Indices for the training set.
    valid_indices (array): Indices for the validation set.
    test_indices (array): Indices for the test set.
    sensor_locations (array): Column indices for sensor data.
    num_sensors (int): Number of signal measurements from sensors (e.g, triaxial = 3)
    m (int): Number of features per timestep
    n (int): Total number of time steps (observations)
    lags (int): length of trajectory
    
  Return:
    test_recons: Reconstructed data from the SHRED model on the test set
    test_ground_truth: Ground truth data from the test set
  """

  all_data_in = np.zeros((n - lags, lags, num_sensors))
  for i in range(len(all_data_in)):
      all_data_in[i] = transformed_X[i:i+lags, sensor_locations]
  ### Generate training validation and test datasets both for reconstruction of states and forecasting sensors
  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
  valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
  test_data_in = torch.tensor(all_data_in[test_indices], dtype=torch.float32).to(device)

  ### -1 to have output be at the same time as final sensor measurements
  train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
  valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
  test_data_out = torch.tensor(transformed_X[test_indices + lags - 1], dtype=torch.float32).to(device)

  train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
  valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
  test_dataset = TimeSeriesDataset(test_data_in, test_data_out)


  ##Modify this part for TCN
  #shred = models_TCN.SHRED_TCN(num_sensors, m, number_channels=2, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
  tcn_chaneels = [64, 64, 64]
  shred = models_TCN.SHRED_TCN(num_sensors, m, num_channels=tcn_chaneels, fc_layers=[350, 400], kernel_size=5, dropout=0.1).to(device)

  validation_errors = models_TCN.fit(shred, train_dataset, valid_dataset, batch_size=64, num_epochs=500, lr=1e-3, verbose=True, patience=3)

  # Generate reconstructions from the test set and print mean square error compared to the ground truth
  test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
  test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())

  return test_recons, test_ground_truth

# Train models

Divide the data into training, validation and test.

In [19]:
# partition into training, validation, test sets
#train_indices, valid_indices, test_indices = ft.partition_data_seq(load_X, n, lags) 
train_indices, valid_indices, _ = ft.partition_data_seq(load_X, n, lags) 
test_data = load_XT


# normalize input data using MinMaxScaler
transformed_X, sc = ft.transform_data(load_X, train_indices) 
transformed_XT = sc.transform(test_data)
test_indices = np.arange(transformed_XT.shape[0])

### Define input sensor

In [20]:
# choose input sensor location
sensor_place = 'RightAnkle' # RightAnkle, Waist, or Chest

# choose input sensor type
sensor_path = '3acc_Training' # 3acc_Training, 3gyro_Training, 3acc3gyro_Training, or Xacc_Training

# access columns indices from main dataframe
sensor_locations, num_sensors = ft.sensor_loc_fun(sensor_path, sensor_place, df_2_data) # This function is specific to the dataset used in this project. Update it according the the types of signals (joint angles, EMG, etc) in your dataset.
train_names = [df_2_data.columns[i] for i in sensor_locations]

print('Number of signals: ', num_sensors)
print('Signals were chosen at: ', sensor_place)
print('Signals chosen: ', [df_2_data.columns[i] for i in sensor_locations])

Number of signals:  3
Signals were chosen at:  RightAnkle
Signals chosen:  ['RightAnkle_Acceleration_x', 'RightAnkle_Acceleration_y', 'RightAnkle_Acceleration_z']


In [21]:
# check path for saving dataframes
if trial_length == 6 and frequency == 128: # full-length trial, full frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ytest_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
else: # reduced trial length or frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'

print(os.path.isdir(save_test_df))
print(save_train_df)
print(save_test_df)

False
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC1214/RightAnkle/3acc_Training/P01_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC1214/RightAnkle/3acc_Training/P01_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv


In [22]:
test_data_length = len(load_XT)
test_time_indices = np.arange(0, test_data_length)
test_times = df_3_data.iloc[test_time_indices,0].to_numpy()

In [23]:
test_times.shape, test_time_indices.shape, load_XT.shape

((38400,), (38400,), (38400, 36))

In [24]:
# train SHRED model
Ypred, Ytest = train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags)

df_Ytest_SHRED = pd.DataFrame(Ytest, columns = df_2_data.columns)
df_Ypred_SHRED = pd.DataFrame(Ypred, columns = df_2_data.columns)

df_Ytest_SHRED['Type']='Measured'
df_Ypred_SHRED['Type']='TCN'

df_Ytest_SHRED['Time']=  test_times          # df_2.iloc[test_indices + lags - 1,0].to_numpy()
df_Ypred_SHRED['Time']=  test_times         # df_2.iloc[test_indices + lags - 1,0].to_numpy()

# save dataframes as .csv if specified
if save_df == True:
  df_Ytest_SHRED.to_csv(save_train_df)
  df_Ypred_SHRED.to_csv(save_test_df)


/mnt/ssd1/wyc/SHREDwyc/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Training epoch 1
Error tensor(0.0970, device='cuda:0')
Training epoch 20
Error tensor(0.0630, device='cuda:0')
Training epoch 40
Error tensor(0.0620, device='cuda:0')
Training epoch 60
Error tensor(0.0621, device='cuda:0')
Training epoch 80
Error tensor(0.0619, device='cuda:0')
Training epoch 100
Error tensor(0.0611, device='cuda:0')
Training epoch 120
Error tensor(0.0609, device='cuda:0')
Training epoch 140
Error tensor(0.0609, device='cuda:0')
Training epoch 160
Error tensor(0.0611, device='cuda:0')
Training epoch 180
Error tensor(0.0610, device='cuda:0')
Training epoch 200
Error tensor(0.0612, device='cuda:0')


# Visualize Results

In [25]:
df_SHRED_tidy = ft.concatRaw(1,sensor_place, sensor_path, 1, 1, df_Ypred_SHRED, df_Ytest_SHRED ,subj)

In [38]:
print("df_SHRED_tidy 的列名:", df_SHRED_tidy.columns)
print("df_SHRED_tidy 的头部数据:\n", df_SHRED_tidy.head(100))

df_SHRED_tidy 的列名: Index(['Subject', 'Pred', 'True', 'Input Location', 'Sensor Type',
       'Output Location', 'Output Signal', 'Output Direction', 'Output Axis',
       'Assessment', 'Time'],
      dtype='object')
df_SHRED_tidy 的头部数据:
    Subject       Pred       True Input Location             Sensor Type  \
0       01  -23.07527  -18.96817     RightAnkle  Triaxial Accelerometer   
1       01  -1.737896   2.577638     RightAnkle  Triaxial Accelerometer   
2       01  -0.920017   2.359172     RightAnkle  Triaxial Accelerometer   
3       01   1.081505   2.237653     RightAnkle  Triaxial Accelerometer   
4       01  -0.077158  -0.298168     RightAnkle  Triaxial Accelerometer   
..     ...        ...        ...            ...                     ...   
95      01   0.280142   0.289567     RightAnkle  Triaxial Accelerometer   
96      01   3.838525   3.960964     RightAnkle  Triaxial Accelerometer   
97      01  36.504608  36.465546     RightAnkle  Triaxial Accelerometer   
98      01 -

In [30]:
# format: ft.extractSignal(output_location, output_signal, output_axis, df_SHRED_tidy)
    # output_location: 'Chest', 'Waist', 'RightAnkle', 'LeftAnkle'
    # output_signal: 'Acceleration', 'Angular Velocity', 'Magnetic Field'
    # output_axis: 'x', 'y', z'

Signal1 = ft.extractSignal('LeftAnkle', 'Acceleration', 'x', df_SHRED_tidy)
Signal2 = ft.extractSignal('Chest', 'Acceleration', 'x', df_SHRED_tidy)
Signal3 = ft.extractSignal('Waist', 'Acceleration', 'x', df_SHRED_tidy)

my_scheme = ['#1e88e5', "#6E6E6E"] # '#014337', '#1e88e5', '#DB1048'

# Compute error: ft.rmse_error, ft.mae_error, OR ft.mbe_error
Signal1_error = ft.rmse_error(Signal1[Signal1['Type'] == 'True']['Value'], Signal1[Signal1['Type'] == 'SHRED']['Value'])
Signal2_error = ft.rmse_error(Signal2[Signal2['Type'] == 'True']['Value'], Signal2[Signal2['Type'] == 'SHRED']['Value'])
Signal3_error = ft.rmse_error(Signal3[Signal3['Type'] == 'True']['Value'], Signal3[Signal3['Type'] == 'SHRED']['Value'])

# plot left ankle acceleration
line1 = alt.Chart(Signal1).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 1 LightAnkle: RMSE = {Signal1_error:.2f}'  # Can change this title to be specific to the output signal
)
# plot chest acceleration
line2 = alt.Chart(Signal2).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 2: Chest RMSE = {Signal2_error:.2f}'  # Can change this title to be specific to the output signal
)

# plot Waist acceleration
line3 = alt.Chart(Signal3).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 3: Waist RMSE = {Signal3_error:.2f}'  # Can change this title to be specific to the output signal
)

final_chart = alt.vconcat(line3, line2, line1).properties(
    title=f'Parameter = {save_tag}, Right Ankle Input' # Can change this title to match the input sensor
    # increase font size
).configure_axis(
    labelFontSize=18,
    titleFontSize=20
).configure_title(
    fontSize=24
).configure_legend(
    labelFontSize=18,
    titleFontSize=20
)

final_chart

alt.VConcatChart(...)

In [28]:
# format: ft.extractSignal(output_location, output_signal, output_axis, df_SHRED_tidy)
    # output_location: 'Chest', 'Waist', 'RightAnkle', 'LeftAnkle'
    # output_signal: 'Acceleration', 'Angular Velocity', 'Magnetic Field'
    # output_axis: 'x', 'y', z'

Signal1 = ft.extractSignal('LeftAnkle', 'Angular Velocity', 'x', df_SHRED_tidy)
Signal2 = ft.extractSignal('Chest', 'Angular Velocity', 'x', df_SHRED_tidy)
Signal3 = ft.extractSignal('Waist', 'Angular Velocity', 'x', df_SHRED_tidy)

my_scheme = ['#1e88e5', "#6E6E6E"] # '#014337', '#1e88e5', '#DB1048'

# Compute error: ft.rmse_error, ft.mae_error, OR ft.mbe_error
Signal1_error = ft.rmse_error(Signal1[Signal1['Type'] == 'True']['Value'], Signal1[Signal1['Type'] == 'SHRED']['Value'])
Signal2_error = ft.rmse_error(Signal2[Signal2['Type'] == 'True']['Value'], Signal2[Signal2['Type'] == 'SHRED']['Value'])
Signal3_error = ft.rmse_error(Signal3[Signal3['Type'] == 'True']['Value'], Signal3[Signal3['Type'] == 'SHRED']['Value'])

# plot left ankle acceleration
line1 = alt.Chart(Signal1).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 1 RightAnkle: RMSE = {Signal1_error:.2f}'  # Can change this title to be specific to the output signal
)
# plot chest acceleration
line2 = alt.Chart(Signal2).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 2: Chest RMSE = {Signal2_error:.2f}'  # Can change this title to be specific to the output signal
)

# plot Waist acceleration
line3 = alt.Chart(Signal3).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 3: Waist RMSE = {Signal3_error:.2f}'  # Can change this title to be specific to the output signal
)

final_chart = alt.vconcat(line3, line2, line1).properties(
    title=f'Parameter = {save_tag}, Right Ankle Input' # Can change this title to match the input sensor
    # increase font size
).configure_axis(
    labelFontSize=18,
    titleFontSize=20
).configure_title(
    fontSize=24
).configure_legend(
    labelFontSize=18,
    titleFontSize=20
)

final_chart

alt.VConcatChart(...)